# Task 4: Convolutional Autoencoder


Restartable model notebook. It loads the saved Task 4 train split and preprocessing configuration from disk.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json
import math

import numpy as np
import optuna
import pandas as pd
import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from torch import nn
from tqdm.auto import tqdm

from src.task4.config import (
    CONFIG_DIR,
    EARLY_STOPPING_PATIENCE,
    FINAL_EPOCHS,
    IMAGE_DIR,
    INPUT_SIZE,
    LABEL_COLUMN,
    LABEL_ID_COLUMN,
    MIN_CLASS_SIZE,
    MIN_DELTA,
    MODEL_DIR,
    N_TRIALS,
    SEED,
    SPLIT_DIR,
    TUNING_EPOCHS,
    VAL_FRACTION,
)
from src.task4.data import make_evaluation_loader, make_model_loaders
from src.task4.models import ConvolutionalAutoencoder, ResNet18Encoder
from src.task4.preprocessing import (
    cae_preprocessing_config,
    cae_transform,
    metric_eval_transform,
    metric_train_transform,
)
from src.task4.retrieval import (
    evaluate_retrieval,
    extract_embeddings,
    k_reciprocal_rerank,
    retrieval_metrics,
    search_cosine,
)
from src.task4.training import (
    AMP_ENABLED,
    DEVICE,
    EarlyStopping,
    cpu_state_dict,
    new_optimizer_and_scheduler,
    set_seed,
    train_one_epoch,
)
from src.task4.tuning import create_study, suggest_parameters


In [3]:
set_seed(SEED)
print("Device:", DEVICE, "| Mixed precision:", AMP_ENABLED)

outer_train_df = pd.read_csv(SPLIT_DIR / "train.csv")
with (CONFIG_DIR / "articleType_gender_label_encoder.json").open(encoding="utf-8") as file:
    label_encoder_config = json.load(file)

classes = label_encoder_config["classes"]
label_to_index = label_encoder_config["label_to_index"]

if LABEL_COLUMN not in outer_train_df.columns:
    outer_train_df[LABEL_COLUMN] = (
        outer_train_df["articleType"].str.strip() + "__" + outer_train_df["gender"].str.strip()
    )
if LABEL_ID_COLUMN not in outer_train_df.columns:
    outer_train_df[LABEL_ID_COLUMN] = outer_train_df[LABEL_COLUMN].map(label_to_index).astype("int64")

outer_class_counts = outer_train_df[LABEL_COLUMN].value_counts()
eligible_labels = set(outer_class_counts[outer_class_counts >= MIN_CLASS_SIZE].index)
development_df = outer_train_df[outer_train_df[LABEL_COLUMN].isin(eligible_labels)].copy()

inner_stratify_columns = ["articleType", "gender"]
inner_stratify_features = pd.get_dummies(
    development_df[inner_stratify_columns].astype(str),
    prefix=inner_stratify_columns,
)
inner_splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=VAL_FRACTION,
    random_state=SEED,
)
inner_train_idx, inner_val_idx = next(
    inner_splitter.split(development_df, inner_stratify_features)
)
train_df = development_df.iloc[inner_train_idx].copy()
val_df = development_df.iloc[inner_val_idx].copy()

overlap = set(train_df["id"]) & set(val_df["id"])
if overlap:
    raise ValueError(f"Inner split contains {len(overlap)} overlapping IDs")
if val_df[LABEL_ID_COLUMN].isna().any():
    raise ValueError("Inner validation contains an unmapped label")

gallery_parts = []
for _, group in train_df.groupby(LABEL_COLUMN, sort=True):
    sample_size = min(20, len(group))
    gallery_parts.append(group.sample(sample_size, random_state=SEED))

tuning_gallery_df = pd.concat(gallery_parts).sort_values("id").reset_index(drop=True)

print(f"Model train/validation: {len(train_df):,}/{len(val_df):,}")
print("Excluded rare rows:", len(outer_train_df) - len(development_df))
print("Tuning gallery rows:", len(tuning_gallery_df))


Device: cuda | Mixed precision: True
Model train/validation: 30,422/3,393
Excluded rare rows: 153
Tuning gallery rows: 2894


In [4]:
def cae_objective(trial: optuna.Trial):
    parameters = suggest_parameters(trial, 'cae')
    set_seed(SEED)
    model = ConvolutionalAutoencoder(ResNet18Encoder())
    loss_function = nn.MSELoss()
    model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
    loaders = make_model_loaders(
        'cae',
        train_df,
        tuning_gallery_df,
        val_df,
        cae_transform,
        metric_train_transform,
        metric_eval_transform,
        IMAGE_DIR,
    )
    optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_score = -math.inf
    for epoch in tqdm(range(1, TUNING_EPOCHS + 1), desc='cae tuning'):
        train_one_epoch('cae', model, loss_function, loaders['train'], optimizer, scaler, epoch)
        score = evaluate_retrieval(model, loaders['gallery'], loaders['query'])['mAP@10']
        scheduler.step(score)
        best_score = max(best_score, score)
        trial.report(score, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_score

In [5]:
cae_study = create_study('task4_cae')
cae_study.optimize(cae_objective, n_trials=N_TRIALS)
cae_best_params = dict(cae_study.best_params)
print('cae', 'best parameters:', cae_best_params)

[I 2026-09-05 20:49:03,940] Using an existing study with name 'task4_cae' instead of creating a new one.


cae tuning:   0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-09-05 20:52:49,251] Trial 9 finished with value: 0.14734454845565956 and parameters: {'learning_rate': 8.468008575248323e-05, 'weight_decay': 0.0007114476009343421}. Best is trial 9 with value: 0.14734454845565956.


cae tuning:   0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-09-05 20:56:44,168] Trial 10 finished with value: 0.13982860110893316 and parameters: {'learning_rate': 0.0006504856968981271, 'weight_decay': 6.251373574521755e-05}. Best is trial 9 with value: 0.14734454845565956.


...

cae tuning:   0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-09-05 21:30:51,114] Trial 27 pruned. 


cae tuning:   0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-09-05 21:32:03,681] Trial 28 pruned. 


cae best parameters: {'learning_rate': 8.468008575248323e-05, 'weight_decay': 0.0007114476009343421}


In [7]:
cae_history = []
cae_best_epoch = 0
cae_best_score = -math.inf

parameters = cae_best_params
set_seed(SEED)
model = ConvolutionalAutoencoder(ResNet18Encoder())
loss_function = nn.MSELoss()
model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
loaders = make_model_loaders(
    "cae",
    train_df,
    tuning_gallery_df,
    val_df,
    cae_transform,
    metric_train_transform,
    metric_eval_transform,
    IMAGE_DIR,
)
optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
stopper = EarlyStopping(EARLY_STOPPING_PATIENCE, MIN_DELTA)
best_state = None
for epoch in tqdm(range(1, FINAL_EPOCHS + 1), desc="cae final"):
    train_loss = train_one_epoch(
        "cae", model, loss_function, loaders["train"], optimizer, scaler, epoch
    )
    metrics = evaluate_retrieval(model, loaders["gallery"], loaders["query"])
    score = metrics["mAP@10"]
    scheduler.step(score)
    if stopper.update(score):
        best_state = cpu_state_dict(model)
        cae_best_epoch = epoch
    cae_history.append({"epoch": epoch, "train_loss": train_loss, **metrics})
    print("cae", epoch, "loss:", round(train_loss, 4), "mAP@10:", round(score, 4))
    if stopper.should_stop:
        break
model.load_state_dict(best_state)
cae_best_score = stopper.best_score
cae_model = model
print("cae", "best epoch:", cae_best_epoch, "best mAP@10:", cae_best_score)


cae final:   0%|          | 0/60 [00:00<?, ?it/s]

cae 1 loss: 0.1109 mAP@10: 0.1204
cae 2 loss: 0.031 mAP@10: 0.1312
cae 3 loss: 0.0136 mAP@10: 0.1325
cae 4 loss: 0.008 mAP@10: 0.136
cae 5 loss: 0.0056 mAP@10: 0.1384
cae 6 loss: 0.0044 mAP@10: 0.14
cae 7 loss: 0.0036 mAP@10: 0.1395
cae 8 loss: 0.0031 mAP@10: 0.1389
cae 9 loss: 0.0027 mAP@10: 0.1429
cae 10 loss: 0.0025 mAP@10: 0.1434
cae 11 loss: 0.0022 mAP@10: 0.1445
cae 12 loss: 0.0021 mAP@10: 0.1454
cae 13 loss: 0.002 mAP@10: 0.1458
cae 14 loss: 0.0019 mAP@10: 0.1469
cae 15 loss: 0.0018 mAP@10: 0.1473
cae 16 loss: 0.0017 mAP@10: 0.146
cae 17 loss: 0.0017 mAP@10: 0.1481
cae 18 loss: 0.0016 mAP@10: 0.1465
cae 19 loss: 0.0015 mAP@10: 0.1452
cae 20 loss: 0.0014 mAP@10: 0.1446
cae 21 loss: 0.0013 mAP@10: 0.1468
cae 22 loss: 0.0013 mAP@10: 0.1473
cae 23 loss: 0.0013 mAP@10: 0.1465
cae 24 loss: 0.0012 mAP@10: 0.1476
cae 25 loss: 0.0012 mAP@10: 0.1472
cae 26 loss: 0.0012 mAP@10: 0.1466
cae 27 loss: 0.0012 mAP@10: 0.1472
cae best epoch: 17 best mAP@10: 0.14813059723021407


In [8]:
model_directory = MODEL_DIR / 'cae'
model_directory.mkdir(parents=True, exist_ok=True)
checkpoint = {
    'model_name': 'cae',
    'model_state_dict': cpu_state_dict(cae_model),
    'model_config': {
        'backbone': 'resnet18', 'pretrained': False,
        'input_size': list(INPUT_SIZE), 'embedding_dimension': 512,
        'class_count': len(classes),
    },
    'best_params': cae_best_params,
    'best_epoch': cae_best_epoch,
    'best_val_map_at_10': cae_best_score,
    'preprocessing_config': cae_preprocessing_config,
    'label_to_index': label_to_index,
    'gallery_scope': 'eligible_inner_training', 'random_seed': SEED,
}
torch.save(checkpoint, model_directory / 'best.pt')
gallery_loader = make_evaluation_loader(train_df, 'cae', cae_transform, metric_eval_transform, IMAGE_DIR)
gallery_embeddings, gallery_ids, gallery_labels = extract_embeddings(cae_model, gallery_loader)
np.save(model_directory / 'gallery_embeddings.npy', gallery_embeddings)
np.save(model_directory / 'gallery_ids.npy', gallery_ids)
query_loader = make_evaluation_loader(val_df, 'cae', cae_transform, metric_eval_transform, IMAGE_DIR)
query_embeddings, _, query_labels = extract_embeddings(cae_model, query_loader)
_, base_indices, index = search_cosine(query_embeddings, gallery_embeddings)
reranked_indices = k_reciprocal_rerank(query_embeddings, gallery_embeddings, index)
print('cae', 'base:', retrieval_metrics(base_indices, query_labels, gallery_labels))
print('cae', 'reranked:', retrieval_metrics(reranked_indices, query_labels, gallery_labels))
print('Saved:', model_directory / 'best.pt')


cae base: {'Precision@1': 0.6934865900383141, 'Recall@1': 0.00373770271461728, 'Precision@5': 0.6224580017683466, 'Recall@5': 0.012514932843105226, 'Precision@10': 0.5834364868847627, 'Recall@10': 0.019932515926476768, 'mAP@10': 0.5041600002806899}
cae reranked: {'Precision@1': 0.5897435897435898, 'Recall@1': 0.0021613954962726566, 'Precision@5': 0.5837901562039494, 'Recall@5': 0.010270391355360635, 'Precision@10': 0.572885352195697, 'Recall@10': 0.01908665422788783, 'mAP@10': 0.49053507688565157}
Saved: /workspace/Machine-Learning-Assignment-2/artifacts/task4/models/cae/best.pt


Observation note: record best validation mAP@10, convergence behavior, and artifact paths here.
